In [1]:
!pip install -q \
torch==2.2.1 \
transformers==4.38.2 \
accelerate==0.27.2 \
sentence-transformers==2.6.1 \
langchain==0.1.16 \
langchain-community==0.0.32 \
langgraph==0.0.32 \
chromadb==0.4.22 \
pypdf \
numpy==1.26.4 \
requests==2.32.4

In [2]:
import torch
print("GPU:", torch.cuda.get_device_name(0))

import transformers, langchain, chromadb
print("Environment ready")

GPU: Tesla T4
Environment ready


In [3]:
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Using:", file_name)

Saving Knowledgebase.pdf to Knowledgebase.pdf
Using: Knowledgebase.pdf


In [4]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = PyPDFLoader(file_name)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

Chunks: 23


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

db = Chroma.from_documents(
    chunks,
    embedding_model
)

print("Vector DB ready ")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possibl

Vector DB ready 


In [6]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=3):
    pairs = [[query, d] for d in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]]

In [7]:
from numpy import dot
from numpy.linalg import norm

def semantic_grounding(answer, context):
    emb1 = embedding_model.embed_query(answer)
    emb2 = embedding_model.embed_query(context[:1000])  # limit size

    similarity = dot(emb1, emb2) / (norm(emb1) * norm(emb2))
    return similarity > 0.55

In [8]:
def retrieve(query, k=5):
    results = db.similarity_search_with_score(query, k=k)

    docs = [doc.page_content for doc, _ in results]
    scores = [score for _, score in results]

    docs = rerank(query, docs, top_k=3)

    return docs, scores

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Phi-2 ready")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Phi-2 ready


In [10]:
def rewrite_query(query):
    prompt = f"""
Rewrite this into a clear, specific question for document search.

Query: {query}
Rewritten:
"""
    return generate_answer(prompt)

In [11]:
def build_prompt(context, query):
    return f"""
You are a STRICT customer support assistant.

Rules:
- Answer ONLY using the given context
- If answer is NOT clearly present → say: "I don't know based on the document"
- Do NOT use outside knowledge
- Be concise and structured

Context:
{context}

Question:
{query}

Answer:
"""

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

In [12]:
def detect_intent(query):
    q = query.lower()

    if any(x in q for x in ["refund", "complaint", "issue", "problem", "not working"]):
        return "complaint"

    elif any(x in q for x in ["summarize", "summary", "overview"]):
        return "summary"

    elif any(x in q for x in ["how", "steps", "process", "procedure"]):
        return "process"

    elif any(x in q for x in ["what", "when", "where", "which"]):
        return "faq"

    return "unknown"


def compute_confidence(scores):
    if not scores:
        return 0.0
    return sum(scores) / len(scores)

In [13]:
def is_answer_grounded(answer, context):
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())

    overlap = len(answer_words.intersection(context_words))

    return overlap / (len(answer_words) + 1) > 0.3


def fallback():
    return "I'm not confident about this answer. Could you rephrase your question based on the document?"

In [14]:
def chat(query):

    try:
        query = rewrite_query(query)
    except:
        pass

    docs, scores = retrieve(query)
    context = " ".join(docs)

    intent = detect_intent(query)
    confidence = compute_confidence(scores)

    if not docs or len(context.strip()) < 50:
        return "[ESCALATED] No relevant information found in document"

    prompt = build_prompt(context, query)
    answer = generate_answer(prompt)

    grounded = semantic_grounding(answer, context)

    if intent == "complaint":
        return "[ESCALATED] Complaint detected → Human support required"

    if not grounded:
        return "I’m not confident this answer is supported by the document. Could you rephrase?"

    if confidence < 0.4:
        return "[ESCALATED] Low confidence retrieval"

    if intent == "unknown":
        return "Can you clarify your question based on the document?"

    response = answer

    response += "\n\n---\n[DEBUG INFO]\n"
    response += f"Intent: {intent}\n"
    response += f"Confidence: {round(confidence,3)}\n"
    response += f"Grounded: {grounded}\n"

    response += "\n[SOURCES]\n"
    for i, doc in enumerate(docs):
        response += f"{i+1}. {doc[:120]}...\n"

    return response

In [15]:
while True:
    query = input("\nAsk: ")

    if query.lower() in ["exit", "quit"]:
        break

    response = chat(query)
    print("\nBot:", response)


Ask: What is this document about?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Bot: I’m not confident this answer is supported by the document. Could you rephrase?

Ask: Summarize the main topic


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Bot: The main topic of the document is the application of VAT/GST based on the billing address jurisdiction, tax exemption certificate submission, password complexity requirements, email verification for account recovery, 2FA issues, and Service Level Agreements (SLAs).

---
[DEBUG INFO]
Intent: faq
Confidence: 1.819
Grounded: True

[SOURCES]
1. VAT/GST is applied based on the billing address jurisdiction. Enterprise customers with tax
exemption certificates must ...
2.  After clicking the link, the customer must enter a new password meeting complexity
requirements: minimum 8 characters,...
3. Professional and Enterprise plans. Starter plans have a 7-day backup retention. Point-in-time
recovery is available for ...


Ask: What is TechFlow Solutions and what does this document contain?


...


Bot: If your account has been hacked, you should immediately create a P0 security incident ticket and notify security@techflow.io. Do not confirm or deny whether a breach has occurred until the Security team has completed their investigation. If the registered email address is no longer accessible, the customer must contact support with government-issued photo ID to verify identity before an account recovery process is initiated.

---
[DEBUG INFO]
Intent: process
Confidence: 1.51
Grounded: True

[SOURCES]
1. 1. Account Management
1.1 Password Reset Procedure
If a customer has forgotten their password, follow these steps to ass...
2. address.
 Processing time: 30 days from verified request receipt.
 After deletion, a confirmation certificate is issue...
3.  After clicking the link, the customer must enter a new password meeting complexity
requirements: minimum 8 characters,...


Ask: Explain this policy


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Bot: I’m not confident this answer is supported by the document. Could you rephrase?

Ask: i need help becouse my account is hacked


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Bot: I’m not confident this answer is supported by the document. Could you rephrase?

Ask: exit
